In [ ]:
import torch

def chat(model, 
         tokenizer, 
         user_message,
         system_message="You are a helpful assistan",
         **Generation_args):
    
    # 1. 역할별 메시지 구성
    # ssytem : 모델의 성격과 규칙, user : "사람의 질문", assistant : 이전 모델 답변
    messages = [
        {"role" : "system", "content" :system_message},
        {"role" : "system", "content" :user_message},
    ]
    
    # 2. Qwen이 학습할 때 쓴 형식으로 변환하기
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    
    # 3. 문자열을 토큰ID 텐서로 바꾸고, gpu로 이동 ( 모델이 있는 곳 )
    inputs = tokenizer(
        text,
    return_tensors="pt").to(model.device)
    
    # 4. 추론이기 때문에 gradient 계산 삭제하기 ( 메모리 절약 )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            ##generation_args
        )
        
        
    # 5. generate는 프롬프트 + 답변을 같이 돌려줌 -> 프롬프트 길이 만큼 잘라내서 답변만 출력함
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]


    # 6. 토큰 ID -> 문자열, <|im_end|> 같은 특수 토큰을 제거   
    return tokenizer.decoder(
        new_tokens,
        skip_special_tokens = True
    )

In [ ]:
# 디코딩 옵션 비교

# Greedy : 매번 확률 1등 토큰만 선택하도록 -> 항상 같은 답이 선택이 됨! 
chat(
    model,
    tokenizer,
    "Recommand a Thai restaurant meal",
    max_new_tokens=80,
    do_sample=False
)


chat(
    model,
    tokenizer,
    "Suggest a dinner menu",
    do_sample=True,
    temperture=0.8,
    top_k=20
)